# CCC by annotator training-set size

This notebook compares individual-annotator CCC as progressively larger and
smaller training-annotation groups are included.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from datasets import load_dataset

# This analysis only needs annotator counts, so load the cached MSP-Podcast
# tables directly rather than initializing the training data pipeline.
with Path('data_config.yaml').open() as config_file:
    data_config = yaml.safe_load(config_file)

cache_dir = Path(data_config['cache_dataset_path']).expanduser()
podcast_version = '2.0'

def load_podcast_split(split):
    path = cache_dir / f'podcastv{podcast_version}_{split}.parquet'
    if not path.is_file():
        raise FileNotFoundError(
            f'Missing cached dataset: {path}. Update cache_dataset_path in '
            'data_config.yaml or generate the dataset cache first.'
        )
    return load_dataset('parquet', data_files={split: str(path)})[split]

# Keep the historical list-of-folds shape used by the remaining cells.
base_train_dataset = [load_podcast_split('train')]
base_test_dataset = [load_podcast_split('test')]


In [ ]:
annotators_to_test_count = {}
for annotators in base_test_dataset[0]['annotators']:
    for annotator in annotators:
        if annotator not in annotators_to_test_count:
            annotators_to_test_count[annotator] = 0
        annotators_to_test_count[annotator] += 1
print(annotators_to_test_count)
annotators_to_train_count = {}
for annotators in base_train_dataset[0]['annotators']:
    for annotator in annotators:
        if annotator not in annotators_to_train_count:
            annotators_to_train_count[annotator] = 0
        annotators_to_train_count[annotator] += 1
print(annotators_to_train_count)

In [ ]:
import pandas as pd
import os
from tqdm import tqdm
from datetime import datetime
import numpy as np

def read_csvs(path):
    # Read all csvs with seeds 
    dated_runs = os.listdir(path)
    if len(dated_runs) > 1:
        print(dated_runs)
        datatimes = [datetime.strptime('_'.join(f.split('_')[-4:]), "%d_%m_%Y_%H:%M:%S") for f in dated_runs]
        most_recent_file = dated_runs[np.argmax(datatimes)]
        # print(files, most_recent_file)
        dated_runs = [most_recent_file]

    assert len(dated_runs) == 1, f'Too many runs, not clear which one to load {dated_runs}'
    results = [pd.read_csv(f'{path}/{dated_runs[0]}/model_prob4_seed_{seed}_fold_0.csv') for seed in range(5)]
    return results

def convert_to_mapping(results):
    mapping = {}
    for _, row in tqdm(results.iterrows()):
        if row['group'].startswith('Annotator') and row['group'].endswith('CCC_ind'):
            act_or_val = 'Activation' if 'Activation' in row['group'] else 'Valence'
            annotator = row['group'].split(' ')[1]
            ccc = row['scalar_value']
            dataset = row['log_type']
            mapping[(annotator, act_or_val, dataset)] = ccc
    return mapping

class CCCIndResultGetter:
    def __init__(self, results_path='.output_files/csv_results/journal_experiments'):
        self.pandas_results = {
            'individual_annotator': [convert_to_mapping(csv) for csv in read_csvs(f'{results_path}/individual_annotator/')],
            'individual_annotator_100plus': [convert_to_mapping(csv) for csv in read_csvs(f'{results_path}/individual_annotator_100plus/')],
            'individual_annotator_1000plus': [convert_to_mapping(csv) for csv in read_csvs(f'{results_path}/individual_annotator_1000plus/')],
        }

    def has_complete_results(self, annotator):
        for model, mappers in self.pandas_results.items():
            dataset = 'Test full-dataset' if model == 'individual_annotator' else 'Test full-dataset similarity'
            for mapper in mappers:
                if any((annotator, dimension, dataset) not in mapper for dimension in ['Activation', 'Valence']):
                    return False
        return True

    def get_ccc_ind_results(self, annotator):
        ind_results = {}
        for model in self.pandas_results:
            valid_datasets = ['Test full-dataset'] if model == 'individual_annotator' else ['Test full-dataset similarity']

            ind_results[model] = {dataset: {
                'act': [float(mapper[(annotator, 'Activation', dataset)]) for mapper in self.pandas_results[model]],
                'val': [float(mapper[(annotator, 'Valence', dataset)]) for mapper in self.pandas_results[model]],
            } for dataset in valid_datasets}
        return ind_results

ccc_ind_result_getter = CCCIndResultGetter()

In [ ]:
all_annotator_cccs = {
    annotator: ccc_ind_result_getter.get_ccc_ind_results(annotator)
    for annotator in tqdm(annotators_to_test_count)
    if annotators_to_test_count[annotator] > 1
    and annotator in annotators_to_train_count
    and ccc_ind_result_getter.has_complete_results(annotator)
}


In [ ]:
# Try plotting cumulative CCC_ind
# i.e. CCC_ind for annotators < 2 annotations, < 3 annotations, etc. until including all annotators 
def get_ccc_ind(annotators):
    seed_results = {}
    
    for annotator in annotators:
        annotator_cccs = all_annotator_cccs[annotator]
        for model in annotator_cccs:
            for dataset in annotator_cccs[model]:
                for seed in range(5):
                    if seed not in seed_results:
                        seed_results[seed] = {}
                    if model not in seed_results[seed]:
                        seed_results[seed][model] = {'act': [], 'val': []}
                    seed_results[seed][model]['act'].append(annotator_cccs[model][dataset]['act'][seed])
                    seed_results[seed][model]['val'].append(annotator_cccs[model][dataset]['val'][seed])
    
    for seed in seed_results:
        for model in seed_results[seed]:
            seed_results[seed][model]['act'] = np.mean(seed_results[seed][model]['act'])
            seed_results[seed][model]['val'] = np.mean(seed_results[seed][model]['val'])

    final_results = {model: {'act': np.mean([seed_results[seed][model]['act'] for seed in seed_results]), 'val': np.mean([seed_results[seed][model]['val'] for seed in seed_results])} for model in seed_results[0]}
    return final_results

x = []
y = {'ia': [], 'ia100': [], 'ia1000': []}
y_val = {'ia': [], 'ia100': [], 'ia1000': []}
max_annotations = max(annotators_to_train_count.values())
annotation_count_to_annotators = {i: [] for i in range(max_annotations + 1)}
for annotator in annotators_to_train_count:
    if annotator not in all_annotator_cccs:
        # print(f'Annotator {annotator} not in test set')
        continue
    annotation_count_to_annotators[annotators_to_train_count[annotator]].append(annotator)

cumulative_annotators = []
up_to_annotations = 75
for i in tqdm(range(0, up_to_annotations, 1)):
    new_annotators = annotation_count_to_annotators[i]
    cumulative_annotators.extend(new_annotators)
    if not len(cumulative_annotators):
        continue
    x.append(i)
    ccc_ind = get_ccc_ind(cumulative_annotators)
    y['ia'].append(ccc_ind['individual_annotator']['act'])
    y_val['ia'].append(ccc_ind['individual_annotator']['val'])
    y['ia100'].append(ccc_ind['individual_annotator_100plus']['act'])
    y_val['ia100'].append(ccc_ind['individual_annotator_100plus']['val'])
    y['ia1000'].append(ccc_ind['individual_annotator_1000plus']['act'])
    y_val['ia1000'].append(ccc_ind['individual_annotator_1000plus']['val'])

plt.plot(x, y['ia'], label='ia')
plt.plot(x, y['ia100'], label='ia100')
plt.plot(x, y['ia1000'], label='ia1000')
plt.legend()
plt.show()
plt.clf()

plt.plot(x, y_val['ia'], label='ia_val')
plt.plot(x, y_val['ia100'], label='ia100_val')
plt.plot(x, y_val['ia1000'], label='ia1000_val')
plt.legend()
plt.show()
plt.clf()

In [ ]:
x = []
xticks = []
y = {'ia': [], 'ia100': [], 'ia1000': []}
y_val = {'ia': [], 'ia100': [], 'ia1000': []}
max_annotations = max(annotators_to_train_count.values())
annotation_count_to_annotators = {i: [] for i in range(max_annotations + 1)}
for annotator in annotators_to_train_count:
    if annotator not in all_annotator_cccs:
        # print(f'Annotator {annotator} not in test set')
        continue
    annotation_count_to_annotators[annotators_to_train_count[annotator]].append(annotator)

up_to_annotations = 1000
cumulative_annotators = []
for i in range(up_to_annotations+1, max(annotation_count_to_annotators.keys())+1):
    new_annotators = annotation_count_to_annotators[i]
    cumulative_annotators.extend(new_annotators)
for i in tqdm(range(up_to_annotations, 0, -1)):
    xticks.append(i)
    new_annotators = annotation_count_to_annotators[i]
    cumulative_annotators.extend(new_annotators)
    if not len(cumulative_annotators):
        continue
    x.append(i)
    ccc_ind = get_ccc_ind(cumulative_annotators)
    y['ia'].append(ccc_ind['individual_annotator']['act'])
    y_val['ia'].append(ccc_ind['individual_annotator']['val'])
    y['ia100'].append(ccc_ind['individual_annotator_100plus']['act'])
    y_val['ia100'].append(ccc_ind['individual_annotator_100plus']['val'])
    y['ia1000'].append(ccc_ind['individual_annotator_1000plus']['act'])
    y_val['ia1000'].append(ccc_ind['individual_annotator_1000plus']['val'])

plt.plot(x, y['ia'], label='ia')
plt.plot(x, y['ia100'], label='ia100')
plt.plot(x, y['ia1000'], label='ia1000')
t = plt.xticks()[0]
plt.xticks(plt.xticks()[0], [int(x) for x in np.linspace(up_to_annotations*1.2, -up_to_annotations*0.2, len(t))])
plt.xlabel('x <= number of annotations')
plt.ylabel('CCC_ind')
plt.legend()
plt.show()
plt.clf()

plt.plot(x, y_val['ia'], label='ia_val')
plt.plot(x, y_val['ia100'], label='ia100_val')
plt.plot(x, y_val['ia1000'], label='ia1000_val')
t = plt.xticks()[0]
plt.xticks(plt.xticks()[0], [int(x) for x in np.linspace(up_to_annotations*1.2, -up_to_annotations*0.2, len(t))])
plt.xlabel('x <= number of annotations')
plt.legend()
plt.show()
plt.clf()